# 🎙️ Pipeline Skripsi VoxCPM & Multi-Generator: Residual & Modulasi Speech Deepfake

Jalankan cell di bawah ini **secara berurutan (Shift + Enter)** dari atas ke bawah.

### 🌟 Fitur Pipeline (Full Google Drive Dataset):
1. **Tabulasi Pemisahan Dataset (`Training` vs `Testing`)**: Cell khusus untuk menyusun dan menampilkan daftar file audio per model generator (`Voxcpm`, `OpenVoice`, `F5TTS`, `E2TTS`) dan pembicara secara rapi.
2. **Inspector Susunan Folder Drive**: Cell khusus untuk melihat struktur folder & nama file audio di Google Drive Anda.
3. **Pemisahan Subfolder `training` vs `testing`**: Data di subfolder `training` digunakan untuk pelatihan model (split 70% Train, 15% Val, 15% Test dengan *data shuffling* dan *speaker split*). Data di subfolder `testing` digunakan khusus sebagai **Dedicated Cross-Generator Blind Test**.
4. **CSV Prediction Mapping Detail**: Menghasilkan pemetaan per file audio (`file_name`, `generator_source`, `y_true`, `y_pred`, `y_score`, `is_correct`).
5. **Visualisasi Confusion Matrix & Metrik Per-Generator**: Menyimpan dan menampilkan matriks kebingungan (PNG & CSV) serta tabel komparasi per generator TTS.

import os
%cd /content

if os.path.exists("skripsi_fase2"):
    %cd skripsi_fase2
    !git fetch origin
    !git reset --hard origin/main
elif os.path.exists(".git"):
    !git fetch origin
    !git reset --hard origin/main
else:
    !git clone https://github.com/alvinrw/skripsi_fase2.git
    %cd skripsi_fase2

!find . -name "*.pyc" -delete 2>/dev/null || true
!find . -name "__pycache__" -delete 2>/dev/null || true

!pip install -q numpy pandas scipy scikit-learn librosa soundfile xgboost pyyaml joblib tqdm statsmodels matplotlib seaborn
print("Setup environment & Sync repository finished!")

In [ ]:
import os
%cd /content

if os.path.exists("skripsi_fase2"):
    %cd skripsi_fase2
    !git pull origin main
elif os.path.exists(".git"):
    !git pull origin main
else:
    !git clone https://github.com/alvinrw/skripsi_fase2.git
    %cd skripsi_fase2

!pip install -q numpy pandas scipy scikit-learn librosa soundfile xgboost pyyaml joblib tqdm statsmodels matplotlib seaborn
print("✅ Setup environment & Pull kode terbaru dari GitHub selesai!")

## 2. Hubungkan Google Drive (Wajib!)
Ini memastikan model dan hasil perhitungan (file CSV & PNG) **tidak hilang** saat Colab ditutup, dan membaca dataset mentah murni dari folder Drive Anda.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/skripsi_results'
for folder in ['results', 'checkpoints', 'manifests', 'figures']:
    os.makedirs(f'{DRIVE_PATH}/{folder}', exist_ok=True)
    if not os.path.islink(folder):
        if os.path.exists(folder):
            !rm -rf {folder}
        os.symlink(f'{DRIVE_PATH}/{folder}', folder)

print(f"✅ Drive terhubung! Semua hasil akan otomatis tersimpan di: {DRIVE_PATH}")

import os
import pandas as pd
from pathlib import Path
from IPython.display import display

DRIVE_DATA_DIR = "/content/drive/MyDrive/Folder_data_inti"
if not os.path.exists(DRIVE_DATA_DIR):
    DRIVE_DATA_DIR = "/content/drive/MyDrive/VoxCPM"

audio_exts = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}
records = []

if os.path.exists(DRIVE_DATA_DIR):
    for root, dirs, files in os.walk(DRIVE_DATA_DIR):
        for f in files:
            if os.path.splitext(f)[1].lower() in audio_exts:
                full_path = os.path.join(root, f).replace("\\", "/")
                parts = full_path.split("/")
                path_clean = full_path.lower().replace("_", "").replace("-", "")
                
                # 1. Deteksi Generator
                if "openvoice" in path_clean: gen = "OpenVoice"
                elif "f5tts" in path_clean: gen = "F5TTS"
                elif "e2tts" in path_clean or "e5tts" in path_clean: gen = "E2TTS"
                elif "voxcpm" in path_clean: gen = "Voxcpm"
                elif any(r in path_clean for r in ["suarareal", "real", "bonafide"]): gen = "Real"
                else: gen = "Other"
                
                # 2. Deteksi Intent (Training vs Testing)
                intent = "Training"
                for p in parts:
                    p_lower = p.lower().replace("_", "").replace("-", "")
                    if "test" in p_lower or "eval" in p_lower:
                        intent = "Testing"
                        break
                    elif "train" in p_lower:
                        intent = "Training"
                        break
                
                # 3. Deteksi Pembicara (Speaker)
                speaker = Path(full_path).parent.name
                if speaker.lower() in ["male", "female", "output_generate", "eval_generate", "f5tts_base", "e2tts_base", "e5tts_base"]:
                    speaker = Path(full_path).parent.parent.name
                    
                records.append({
                    "generator": gen,
                    "intent": intent,
                    "speaker": speaker,
                    "file_name": f,
                    "file_path": full_path
                })

df_all = pd.DataFrame(records)

if not df_all.empty:
    print("==================================================================================")
    print("🏋️ TABEL 1: RINGKASAN DATASET TRAINING (Per Model Generator & Pembicara):")
    print("==========================================================================")
    df_train = df_all[df_all["intent"] == "Training"]
    df_train_sum = df_train.groupby(["generator", "speaker"])["file_name"].count().reset_index()
    df_train_sum.columns = ["Model Generator", "Nama Pembicara (Speaker)", "Jumlah File Audio"]
    display(df_train_sum)
    print(f"Total File Audio Training: {len(df_train)} file")
    
    print("\n==========================================================================")
    print("🧪 TABEL 2: RINGKASAN DATASET TESTING (Per Model Generator & Pembicara):")
    print("==========================================================================")
    df_test = df_all[df_all["intent"] == "Testing"]
    df_test_sum = df_test.groupby(["generator", "speaker"])["file_name"].count().reset_index()
    df_test_sum.columns = ["Model Generator", "Nama Pembicara (Speaker)", "Jumlah File Audio"]
    display(df_test_sum)
    print(f"Total File Audio Testing (Blind Test): {len(df_test)} file")
else:
    print("⚠️ File audio belum ditemukan di Google Drive.")

In [ ]:
import os

DRIVE_DATA_DIR = "/content/drive/MyDrive/Folder_data_inti"
if not os.path.exists(DRIVE_DATA_DIR):
    DRIVE_DATA_DIR = "/content/drive/MyDrive/VoxCPM"

print(f"📁 SUSUNAN DIREKTORI GOOGLE DRIVE:")
print(f"Path: {DRIVE_DATA_DIR}\n" + "="*60)

if os.path.exists(DRIVE_DATA_DIR):
    audio_exts = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}
    for root, dirs, files in os.walk(DRIVE_DATA_DIR):
        level = root.replace(DRIVE_DATA_DIR, '').count(os.sep)
        indent = ' ' * 4 * level
        folder_name = os.path.basename(root) if os.path.basename(root) else DRIVE_DATA_DIR
        audio_files = [f for f in files if os.path.splitext(f)[1].lower() in audio_exts]
        print(f"{indent}📂 {folder_name}/ ({len(audio_files)} file audio)")
        
        if audio_files:
            for sample in audio_files[:3]:
                print(f"{indent}    📄 {sample}")
            if len(audio_files) > 3:
                print(f"{indent}    ... dan {len(audio_files)-3} file audio lainnya")
else:
    print(f"⚠️ Folder {DRIVE_DATA_DIR} belum ditemukan. Pastikan Google Drive sudah di-mount.")

## 2.6. Tabulasi & Pemetaan Dataset (TRAINING vs TESTING per Model Generator) 📋📊
Cell ini secara otomatis mengekstrak seluruh file audio dari struktur berantakan di Google Drive dan menampilkannya ke dalam **Tabel Rapi**: memisahkan dataset **TRAINING** (Voxcpm, OpenVoice, F5TTS, E2TTS) dan dataset **TESTING** (Voxcpm, OpenVoice, F5TTS, E2TTS) beserta pembicaranya.

In [ ]:
import os
import pandas as pd
from pathlib import Path
from IPython.display import display

DRIVE_DATA_DIR = "/content/drive/MyDrive/Folder_data_inti"
if not os.path.exists(DRIVE_DATA_DIR):
    DRIVE_DATA_DIR = "/content/drive/MyDrive/VoxCPM"

audio_exts = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}
records = []

if os.path.exists(DRIVE_DATA_DIR):
    for root, dirs, files in os.walk(DRIVE_DATA_DIR):
        for f in files:
            if os.path.splitext(f)[1].lower() in audio_exts:
                full_path = os.path.join(root, f).replace("\\", "/")
                parts = full_path.split("/")
                path_clean = full_path.lower().replace("_", "").replace("-", "")
                
                # 1. Deteksi Generator
                if "openvoice" in path_clean: gen = "OpenVoice"
                elif "f5tts" in path_clean: gen = "F5TTS"
                elif "e2tts" in path_clean: gen = "E2TTS"
                elif "voxcpm" in path_clean: gen = "Voxcpm"
                elif any(r in path_clean for r in ["suarareal", "real", "bonafide"]): gen = "Real"
                else: gen = "Other"
                
                # 2. Deteksi Intent (Training vs Testing)
                intent = "Training"
                for p in parts:
                    p_lower = p.lower().replace("_", "").replace("-", "")
                    if "test" in p_lower or "eval" in p_lower:
                        intent = "Testing"
                        break
                    elif "train" in p_lower:
                        intent = "Training"
                        break
                
                # 3. Deteksi Pembicara (Speaker)
                speaker = Path(full_path).parent.name
                if speaker.lower() in ["male", "female", "output_generate", "f5tts_base", "e2tts_base"]:
                    speaker = Path(full_path).parent.parent.name
                    
                records.append({
                    "generator": gen,
                    "intent": intent,
                    "speaker": speaker,
                    "file_name": f,
                    "file_path": full_path
                })

df_all = pd.DataFrame(records)

if not df_all.empty:
    print("==================================================================================")
    print("🏋️ TABEL 1: RINGKASAN DATASET TRAINING (Per Model Generator & Pembicara):")
    print("==========================================================================")
    df_train = df_all[df_all["intent"] == "Training"]
    df_train_sum = df_train.groupby(["generator", "speaker"])["file_name"].count().reset_index()
    df_train_sum.columns = ["Model Generator", "Nama Pembicara (Speaker)", "Jumlah File Audio"]
    display(df_train_sum)
    print(f"Total File Audio Training: {len(df_train)} file")
    
    print("\n==========================================================================")
    print("🧪 TABEL 2: RINGKASAN DATASET TESTING (Per Model Generator & Pembicara):")
    print("==========================================================================")
    df_test = df_all[df_all["intent"] == "Testing"]
    df_test_sum = df_test.groupby(["generator", "speaker"])["file_name"].count().reset_index()
    df_test_sum.columns = ["Model Generator", "Nama Pembicara (Speaker)", "Jumlah File Audio"]
    display(df_test_sum)
    print(f"Total File Audio Testing (Blind Test): {len(df_test)} file")
else:
    print("⚠️ File audio belum ditemukan di Google Drive.")

## 3. Persiapan & QC Dataset (`Folder_data_inti`)
Melakukan scanning folder generator murni dari Google Drive Anda (`Voxcpm`, `OpenVoice`, `F5TTS`, `E2TTS`), mendeteksi subfolder `training` vs `testing`, melakukan QC audio, pemotongan (*chunking*) 2 detik, serta pemisahan manifest.

*(Sesuaikan `DRIVE_DATA_DIR` jika lokasi folder data utama Anda berada di tempat lain)*

In [ ]:
import os
if os.path.exists("/content/skripsi_fase2"):
    %cd /content/skripsi_fase2

# Tentukan lokasi folder data utama Anda di Google Drive
DRIVE_DATA_DIR = "/content/drive/MyDrive/Folder_data_inti"

if not os.path.exists(DRIVE_DATA_DIR):
    DRIVE_DATA_DIR = "/content/drive/MyDrive/VoxCPM"

print(f"📂 Menggunakan lokasi dataset: {DRIVE_DATA_DIR}")

# Jalankan persiapan dataset murni dari Google Drive
!python src/run_pipeline.py --steps prepare --drive_dir "{DRIVE_DATA_DIR}" --out_dir "/content/VoxCPM_processed"
print("✅ Persiapan dataset selesai!")

## 4. Eksekusi Penuh Pipeline Skripsi 🔥
Ekstraksi fitur MFCC/LFCC/Residual/Modulasi, pelatihan model Machine Learning pada dataset *training* (dengan *data shuffling*), serta pengujian pada **In-Domain Test Set** dan **Dedicated Separate Testing Set**.

> ⏳ **Catatan:** Ekstraksi fitur dan *training* akan memakan waktu tergantung jumlah file audio.

In [ ]:
import os
if os.path.exists("/content/skripsi_fase2"):
    %cd /content/skripsi_fase2

!python src/run_pipeline.py --steps features --duration 2s
!python src/run_pipeline.py --steps train --duration 2s
!python src/run_pipeline.py --steps stats consistency bootstrap evaluate --duration 2s
print('✅ Eksekusi pipeline durasi 2s selesai!')

## 5. Inspeksi CSV Prediction Mapping & Confusion Matrices 📊
Menampilkan preview hasil klasifikasi per file audio, per-generator breakdown (Voxcpm vs OpenVoice vs F5TTS vs E2TTS), serta gambar Confusion Matrix.

In [ ]:
import os
if os.path.exists("/content/skripsi_fase2"):
    %cd /content/skripsi_fase2

import pandas as pd
import glob
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

# 1. Ringkasan Metrik Pengujian
metrics_path = 'results/metrics_2s.csv'
if os.path.exists(metrics_path):
    df_m = pd.read_csv(metrics_path)
    print("🏆 PERBANDINGAN PERFORMA MODEL (Validation vs Test vs Separate Test):")
    display(df_m[['model', 'split', 'auc', 'eer', 'accuracy', 'f1_macro']])
else:
    print("⚠️ File metrics belum ditemukan.")

# 2. Preview CSV Prediction Mapping Terpisah (Separate Blind Test)
mapping_files = sorted(glob.glob('results/prediction_mapping_*_separate_test.csv'))
if mapping_files:
    print("\n📄 PREVIEW CSV PREDICTION MAPPING (Separate Blind Test):")
    df_map = pd.read_csv(mapping_files[0])
    display(df_map.head(10)[['file_name', 'generator_source', 'true_label', 'predicted_label', 'confidence_score_fake', 'is_correct']])
    
# 3. Breakdown Metrics Per Generator TTS
gen_files = sorted(glob.glob('results/per_generator_metrics_*_separate_test.csv'))
if gen_files:
    print("\n📊 BREAKDOWN PERFORMA PER-GENERATOR TTS (Voxcpm vs OpenVoice vs F5TTS vs E2TTS):")
    df_gen = pd.read_csv(gen_files[0])
    display(df_gen)

# 4. Visualisasi Confusion Matrix PNG
cm_pngs = sorted(glob.glob('results/confusion_matrices/*.png'))
if cm_pngs:
    print("\n🖼️ CONFUSION MATRICES (Top 3 Model):")
    for p in cm_pngs[:3]:
        print(f"File: {p}")
        display(Image(filename=p))


## 🎉 Selesai!
Semua proses telah tuntas. Seluruh file **CSV Mapping**, **Per-Generator Breakdown**, **Confusion Matrix PNG & CSV**, serta **Checkpoint Model** telah otomatis tersimpan di Google Drive Anda (`/content/drive/MyDrive/skripsi_results`). Siap dipakai langsung di naskah skripsi Anda!